# Remove Test Leakage from Train
Menghapus gambar test yang bocor ke folder train (data leakage).

> **Note:** Jalankan setelah `data_quality.ipynb` selesai (backup & duplicate cleaning sudah dilakukan).
> Backup train ada di `data/raw/train_backup/` (26.527 gambar, sebelum cleaning).

In [ ]:
import hashlib
import os
from collections import defaultdict
from pathlib import Path

def _safe_path(p):
    p = str(p)
    if len(p) > 240 and not p.startswith('\\\\?\\'):
        return '\\\\?\\' + p
    return p

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train'
TEST_DIR = PROJECT_ROOT / 'data' / 'raw' / 'test'
BACKUP_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_backup'

print(f'Project root: {PROJECT_ROOT}')
print(f'Train dir: {TRAIN_DIR}')
print(f'Test dir: {TEST_DIR}')
print(f'Backup dir: {BACKUP_DIR} (exists: {BACKUP_DIR.exists()})')

---
## 1. Scan MD5 Test Images

In [ ]:
test_hashes = {}
for fpath in TEST_DIR.iterdir():
    with open(_safe_path(fpath), 'rb') as f:
        h = hashlib.md5(f.read()).hexdigest()
    test_hashes[h] = fpath.name

print(f'Test images scanned: {len(test_hashes)}')

---
## 2. Scan MD5 Train Images

In [ ]:
train_hashes = defaultdict(list)
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = TRAIN_DIR / cls
    if not cls_path.is_dir():
        continue
    for fpath in cls_path.iterdir():
        with open(_safe_path(fpath), 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()
        train_hashes[h].append({'cls': cls, 'fname': fpath.name, 'path': fpath})

total_train = sum(len(v) for v in train_hashes.values())
print(f'Train images scanned: {total_train} ({len(train_hashes)} unique)')

---
## 3. Cari Overlap (Data Leakage)

In [ ]:
overlap_hashes = set(test_hashes.keys()) & set(train_hashes.keys())
leaked_files = []

if not overlap_hashes:
    print('No overlap found. Data already clean! \u2705')
else:
    print(f'Found {len(overlap_hashes)} overlapping images:')
    for h in sorted(overlap_hashes):
        test_name = test_hashes[h]
        for tf in train_hashes[h]:
            leaked_files.append(tf)
            print(f'  Test: {test_name} === Train: {tf["cls"]}/{tf["fname"]}')

    # Count per class
    per_class = defaultdict(int)
    for tf in leaked_files:
        per_class[tf['cls']] += 1

    print()
    print('Per class:')
    for cls, cnt in sorted(per_class.items()):
        print(f'  {cls}: {cnt}')
    print(f'  TOTAL: {len(leaked_files)}')

---
## 4. Hapus Leaked Files dari Train
> Backup sudah ada di `train_backup/` (dibuat oleh `data_quality.ipynb`).

In [ ]:
if not overlap_hashes:
    print('Nothing to remove. Data is clean!')
else:
    confirm = input(f'Yakin mau hapus {len(leaked_files)} leaked files dari train? (y/n): ').strip().lower()
    if confirm != 'y':
        print('Dibatalkan.')
    else:
        for tf in leaked_files:
            os.remove(tf['path'])
            print(f'  REMOVED: {tf["cls"]}/{tf["fname"]}')
        print(f'\nDone! Removed {len(leaked_files)} leaked files.')

---
## 5. Verifikasi Final

In [ ]:
final_counts = {}
total_final = 0
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = TRAIN_DIR / cls
    if not cls_path.is_dir():
        continue
    count = len(list(cls_path.iterdir()))
    final_counts[cls] = count
    total_final += count

print('Final Class Distribution (after leakage removal):')
for cls, count in sorted(final_counts.items()):
    print(f'  {cls:<25} {count:>8}')
print('  ' + '-' * 35)
print(f'  {"TOTAL":<25} {total_final:>8}')